# GOaT selection notebook

Run top to bottom on Colab. Every stage writes its artifacts to Google Drive
and skips itself when those artifacts already exist.

## Step 1. Open the Colab notebook

## Step 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Step 3. Git clone the repo, shallow

In [ ]:
![ -d /content/GOaT/.git ] || git clone --depth 1 https://github.com/champyod/GOaT.git /content/GOaT
!ls -lh /content/GOaT/model/src/goat_model 2>&1 | head -n 20
!ls -lh /content/GOaT/model/notebooks 2>&1 | head -n 20

## Step 4. Install dependencies

In [ ]:
!apt-get install -y -q libraqm0 > /dev/null 2>&1
# Single install source: model/requirements.txt (mirrors pyproject extras).
# Primary (laptop once): colab install -s goat -r requirements.txt.
# Fallback (direct in-notebook, runs when colab install skipped):
%pip install -q -r /content/GOaT/model/requirements.txt


## Step 5. Import the package from the clone

In [ ]:
import sys
sys.path.insert(0, "/content/GOaT/model/src")
from goat_model import constants as c
print("goat_model at", c.__file__)

## Args (Drive — passed to every step)

In [ ]:
DRIVE = str(c.DRIVE_ROOT)
MT_DATA = str(c.DRIVE_PATHS["mt"])
MT_TEST_DIR = str(c.DRIVE_PATHS["mt_test"])
OCR_EVAL_DIR = str(c.DRIVE_PATHS["ocr_eval"])
RESULTS = str(c.DRIVE_PATHS["results"])
DATA_ROOT = str(c.DRIVE_PATHS["data_root"])
ART_MT = str(c.DRIVE_PATHS["art_mt"])
ART_OCR = str(c.DRIVE_PATHS["art_ocr"])
SEED = c.SEED
REPEATS_MT = c.MT_N_RUNS
REPEATS_OCR = c.OCR_N_RUNS


## Step 6. Data — download

In [ ]:
%run /content/GOaT/model/notebooks/data/download_data.py --dataset scb-mt --out-dir $MT_DATA
%run /content/GOaT/model/notebooks/data/download_data.py --dataset flores200 --out-dir $MT_TEST_DIR
%run /content/GOaT/model/notebooks/data/download_data.py --dataset thaiocrbench --out-dir $OCR_EVAL_DIR/thaiocrbench
%run /content/GOaT/model/notebooks/data/download_data.py --dataset thai-ocr-evaluation --out-dir $OCR_EVAL_DIR/thai-ocr-evaluation


## Step 7. Selection MT

In [ ]:
%run /content/GOaT/model/notebooks/selection/select_mt.py --mt-test-dir $MT_TEST_DIR --output $RESULTS/mt_selection.json --repeats $REPEATS_MT --seed $SEED


## Step 8. Selection OCR

In [ ]:
%run /content/GOaT/model/notebooks/selection/select_ocr.py --ocr-eval-dir $OCR_EVAL_DIR --output $RESULTS/ocr_selection.json --repeats $REPEATS_OCR --seed $SEED
